In [1]:
import time

import pandas as pd

from src.data.load_raw import load_raw_data
from src.data.preprocess import (
    extract_xy,
    merge_features_and_labels,
    standardize_splits,
)
from src.data.split import split_labeled_transactions
from src.evaluation.metrics import calculate_binary_metrics
from src.evaluation.threshold import find_best_f1_threshold
from src.models.classic import (
    build_dummy_classifier,
    build_logistic_regression,
    build_random_forest,
    build_xgboost,
)

In [2]:
features_df, classes_df, edges_df = load_raw_data()

transactions_df = merge_features_and_labels(
    features_df,
    classes_df,
)

splits = split_labeled_transactions(transactions_df)

print("Features:", features_df.shape)
print("Classes:", classes_df.shape)
print("Edges:", edges_df.shape)

print("\nTrain:", len(splits.train))
print("Validation:", len(splits.validation))
print("Test:", len(splits.test))

print("\nTrain labels:")
print(splits.train["target"].value_counts())

print("\nValidation labels:")
print(splits.validation["target"].value_counts())

print("\nTest labels:")
print(splits.test["target"].value_counts())

Features: (203769, 167)
Classes: (203769, 2)
Edges: (234355, 2)

Train: 29894
Validation: 7829
Test: 8841

Train labels:
target
0    26432
1     3462
Name: count, dtype: Int64

Validation labels:
target
0    7154
1     675
Name: count, dtype: Int64

Test labels:
target
0    8433
1     408
Name: count, dtype: Int64


In [3]:
x_train, y_train = extract_xy(
    splits.train,
    feature_set="local",
)

x_validation, y_validation = extract_xy(
    splits.validation,
    feature_set="local",
)

x_test, y_test = extract_xy(
    splits.test,
    feature_set="local",
)

(
    scaler,
    x_train_scaled,
    x_validation_scaled,
    x_test_scaled,
) = standardize_splits(
    x_train,
    x_validation,
    x_test,
)

In [4]:
results = []


def run_experiment(
    model_name,
    model,
    x_train,
    y_train,
    x_validation,
    y_validation,
    x_test,
    y_test,
):
    start_time = time.perf_counter()

    model.fit(
        x_train,
        y_train,
    )

    training_time = time.perf_counter() - start_time

    validation_scores = model.predict_proba(
        x_validation,
    )[:, 1]

    threshold = find_best_f1_threshold(
        y_validation,
        validation_scores,
    )

    test_scores = model.predict_proba(
        x_test,
    )[:, 1]

    metrics = calculate_binary_metrics(
        y_test,
        test_scores,
        threshold=threshold,
    )

    metrics.update(
        {
            "model": model_name,
            "feature_set": "local",
            "training_time_seconds": training_time,
        }
    )

    results.append(metrics)

    return model, metrics

In [5]:
dummy_model, dummy_metrics = run_experiment(
    model_name="Dummy Classifier",
    model=build_dummy_classifier(),
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
    x_test=x_test,
    y_test=y_test
)

In [6]:
logistic_model, logistic_metrics = run_experiment(
    model_name="Logistic Regression",
    model=build_logistic_regression(solver='lbfgs', penalty='l2', C=101.85509528793979, max_iter=800),
    x_train=x_train_scaled,
    y_train=y_train,
    x_validation=x_validation_scaled,
    y_validation=y_validation,
    x_test=x_test_scaled,
    y_test=y_test
)


/home/piotrs/bitcoin-transaction-classification/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [7]:
random_forest_model, random_forest_metrics = run_experiment(
    model_name="Random Forest",
    model=build_random_forest(n_estimators=1000, max_depth=21, min_samples_split=12, min_samples_leaf=4, max_features='log2', criterion='entropy'),
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
    x_test=x_test,
    y_test=y_test
)

In [8]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

scale_pos_weight = negative_count / positive_count

print("scale_pos_weight:", scale_pos_weight)

xgboost_model, xgboost_metrics = run_experiment(
    model_name="XGBoost",
    model=build_xgboost(
        #scale_pos_weight=scale_pos_weight,
        learning_rate=0.03555882585346509,
        n_estimators=100,
        max_depth=12,
        min_child_weight=6,
        gamma=0.9341454822832975,
        subsample=0.5601002934661604,
        colsample_bytree=0.528185149633782,
        scale_pos_weight=14.814059852192749
    ),
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
    x_test=x_test,
    y_test=y_test,
)

scale_pos_weight: 7.634893125361063


In [9]:
def run_feature_set_experiment(
    feature_set,
    model_name,
    model,
):
    x_train, y_train = extract_xy(
        splits.train,
        feature_set=feature_set,
    )

    x_validation, y_validation = extract_xy(
        splits.validation,
        feature_set=feature_set,
    )

    x_test, y_test = extract_xy(
        splits.test,
        feature_set=feature_set,
    )

    start_time = time.perf_counter()

    model.fit(x_train, y_train)

    training_time = time.perf_counter() - start_time

    validation_scores = model.predict_proba(
        x_validation,
    )[:, 1]

    threshold = find_best_f1_threshold(
        y_validation,
        validation_scores,
    )

    test_scores = model.predict_proba(
        x_test,
    )[:, 1]

    metrics = calculate_binary_metrics(
        y_test,
        test_scores,
        threshold=threshold,
    )

    metrics.update(
        {
            "model": model_name,
            "feature_set": feature_set,
            "training_time_seconds": training_time,
        }
    )

    results.append(metrics)

    return model, metrics

In [10]:
lr_all_model, lr_all_metrics = run_feature_set_experiment(
    feature_set="all",
    model_name="Logistic Regression",
    model=build_logistic_regression(
        # solver='lbfgs',
        # penalty=None,
        # max_iter=1200
        solver='lbfgs', penalty='l2', C=101.85509528793979, max_iter=800
    ),
)

rf_all_model, rf_all_metrics = run_feature_set_experiment(
    feature_set="all",
    model_name="Random Forest",
    model=build_random_forest(n_estimators=50, max_features=50)
)

xgb_all_model, xgb_all_metrics = run_feature_set_experiment(
    feature_set="all",
    model_name="XGBoost",
    model=build_xgboost(
        # learning_rate=0.003074175958275189,
        # n_estimators=100,
        # max_depth=11,
        # min_child_weight=9,
        # subsample=0.6090369067334965,
        # colsample_bytree=0.9662933451942084,
        # gamma=2.304436293405026,
        # reg_alpha=4.007437246657634e-05,
        # reg_lambda=0.7025980714477842,
        # scale_pos_weight=4.7386556285358195
        # learning_rate=0.03555882585346509,
        # n_estimators=100,
        # max_depth=12,
        # min_child_weight=6,
        # gamma=0.9341454822832975,
        # subsample=0.5601002934661604,
        # colsample_bytree=0.528185149633782,
        scale_pos_weight=scale_pos_weight
    )
)

/home/piotrs/bitcoin-transaction-classification/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/piotrs/bitcoin-transaction-classification/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 800 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=800).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternati

In [12]:
xgb_all_model, xgb_all_metrics = run_feature_set_experiment(
    feature_set="all",
    model_name="XGBoost2",
    model=build_xgboost(
        # learning_rate=0.003074175958275189,
        # n_estimators=100,
        # max_depth=11,
        # min_child_weight=9,
        # subsample=0.6090369067334965,
        # colsample_bytree=0.9662933451942084,
        # gamma=2.304436293405026,
        # reg_alpha=4.007437246657634e-05,
        # reg_lambda=0.7025980714477842,
        # scale_pos_weight=4.7386556285358195
        # learning_rate=0.03555882585346509,
        # n_estimators=100,
        # max_depth=12,
        # min_child_weight=6,
        # gamma=0.9341454822832975,
        # subsample=0.5601002934661604,
        # colsample_bytree=0.528185149633782,
        scale_pos_weight=scale_pos_weight
    )
)

In [13]:
results_df = pd.DataFrame(results)

columns = [
    "model",
    "feature_set",
    "pr_auc",
    "roc_auc",
    "precision_illicit",
    "recall_illicit",
    "f1_illicit",
    "threshold",
    "training_time_seconds",
]

results_df[columns].sort_values(
    by="pr_auc",
    ascending=False,
).style.format(
    {
        "pr_auc": "{:.4f}",
        "roc_auc": "{:.4f}",
        "precision_illicit": "{:.4f}",
        "recall_illicit": "{:.4f}",
        "f1_illicit": "{:.4f}",
        "threshold": "{:.2f}",
        "training_time_seconds": "{:.2f}",
    }
)

display(results_df)

,pr_auc,roc_auc,precision_illicit,recall_illicit,f1_illicit,confusion_matrix,threshold,model,feature_set,training_time_seconds
0,0.046149,0.500000,0.046149,1.000000,0.088226,"[[0, 8433], [0, 408]]",0.01,Dummy Classifier,local,0.002525
1,0.249995,0.821980,0.515493,0.448529,0.479685,"[[8261, 172], [225, 183]]",0.77,Logistic Regression,local,31.993827
2,0.535525,0.817903,0.874419,0.460784,0.603531,"[[8406, 27], [220, 188]]",0.60,Random Forest,local,3.874098
3,0.536629,0.815021,0.776423,0.468137,0.584098,"[[8378, 55], [217, 191]]",0.86,XGBoost,local,1.628992
4,0.242512,0.839639,0.410753,0.468137,0.437572,"[[8159, 274], [217, 191]]",0.69,Logistic Regression,all,62.897048
5,0.537766,0.852745,0.926471,0.463235,0.617647,"[[8418, 15], [219, 189]]",0.49,Random Forest,all,1.378426
6,0.549091,0.861435,0.892523,0.468137,0.614148,"[[8410, 23], [217, 191]]",0.82,XGBoost,all,3.177039
7,0.556876,0.856417,0.923445,0.473039,0.625608,"[[8417, 16], [215, 193]]",0.86,XGBoost2,all,3.267910
